In [1]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'test-download-capitanata'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Mounted at /content/drive
Cloning into '/content/crop-spatial-classification'...
remote: Enumerating objects: 631, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 631 (delta 81), reused 84 (delta 64), pack-reused 524 (from 1)
Receiving objects: 100% (631/631), 197.98 MiB | 39.84 MiB/s, done.
Resolving deltas: 100% (424/424), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 1.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.9 MB/s eta 0:00:00
Setup ambiente Colab completato! Branch attivo su Colab:  test-download-capitanata
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/MyDrive/Progetto MLDM/data/raw


# 01 - Estrazione del ground truth e campionamento delle colture
estrae un campione bilanciato di punti per ciascun tipo di coltura e genera la legenda associata.

#### Ricerca dei file TIF per ciascun anno

In [2]:
from pathlib import Path

# anno/i di estrazione dei dati
TARGET_YEARS = ["2023"]

main_directory = Path(f"{DATA_DIR}/raw/crops_types_yearly_capitanata_03035")
tifs_3035 = {}

for year in TARGET_YEARS:
    year_dir = main_directory / year
    if year_dir.is_dir():
        # trova tutti i .tif per questo anno
        tifs_3035[year] = [str(tif) for tif in year_dir.rglob("*.tif")]
        print(f"Anno {year}: trovati {len(tifs_3035[year])} file .tif")
    else:
        print(f"Cartella anno {year} non trovata in {main_directory}")

Anno 2023: trovati 4 file .tif


#### Riproiezione in coordinate GPS (EPSG:4326)

In [3]:
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

tifs_4326 = {}

for year, file_paths in tifs_3035.items():
    year_file_list = []

    for row_path in file_paths:
        # costruisce il percorso del file riproiettato in EPSG:4326
        file_name = row_path.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # apre il file GeoTIFF originale (EPSG:3035)
        raster_3035 = rioxarray.open_rasterio(row_path)

        # riproiezione in EPSG:4326, usando NEAREST per preservare i codici delle colture
        raster_4326 = raster_3035.rio.reproject("EPSG:4326", resampling=Resampling.nearest)

        # salva il file riproiettato
        raster_4326.rio.to_raster(file_name)
        raster_3035.close()

        year_file_list.append(file_name)
        
    tifs_4326[year] = year_file_list

print(f"\nRiproiezione completata per {sum(len(v) for v in tifs_4326.values())} file")


Riproiezione completata per 4 file


#### Estrazione di un campione bilanciato di classi di colture

In [ ]:
import json
from pathlib import Path
import rasterio
import numpy as np
from src.config import is_in_capitanata, EXCLUDED_CROP_CLASSES

# num. max di punti casuali da estrarre per ciascuna coltura da ogni file .tif
SAMPLES_PER_CLASS_PER_TIF = 50

points_path = DATA_DIR / 'interim' / 'points.json'

# se points.json esiste già, tiene quelli già salvati e ne aggiunge eventualmente di nuovi 
if points_path.exists():
    with open(points_path, "r", encoding="utf-8") as f:
        points = json.load(f)
    print(f"Trovati {len(points)} punti già esistenti. Aggiunta di nuovi punti in corso...")
else:
    points = []

# crea un set con le coordinate già salvate per non estrarre doppioni
existing_coordinates = {(round(p["lat"], 5), round(p["lon"], 5)) for p in points}

# N.B. garantisce di ottenere gli stessi risultati casuali ad ogni esecuzione
np.random.seed(42)

# itera su tutti gli anni target e sui rispettivi .tif riproiettati
for year, file_list in tifs_4326.items():
    print(f"Campionamento in corso per l'anno {year} ({len(file_list)} file)")

    for tif_path in file_list:
        with rasterio.open(tif_path) as dataset:
            # legge la matrice 2D dei pixel (banda 1)
            crop_matrix = dataset.read(1)

            # crea una maschera per scartare i pixel nulli o nodata
            valid_mask = (crop_matrix > 0) & (crop_matrix < 65534)

            # trova l'elenco dei codici coltura presenti nella matrice
            crop_codes = np.unique(crop_matrix[valid_mask])

            for crop_code in crop_codes:
                # scarta le classi da escludere
                if int(crop_code) in EXCLUDED_CROP_CLASSES:
                    continue

                # seleziona solo i pixel di questa coltura
                rows, cols = np.where(crop_matrix == crop_code)

                # verifica quanti punti di questa coltura ci sono già per questo file
                # ed estrae solo la differenza per raggiungere la nuova quota (SAMPLES_PER_CLASS_PER_TIF) 
                indices = np.random.choice(len(rows), size=len(rows), replace=False)

                added_for_this_class = 0
                for index in indices:
                    if added_for_this_class >= SAMPLES_PER_CLASS_PER_TIF:
                        break

                    r, c = rows[index], cols[index]
                    lon, lat = dataset.xy(r, c)

                    # scarta il campo se si trova fuori dai confini della Capitanata
                    if not is_in_capitanata(lon, lat):
                        continue

                    # converte i pixel del campo in coordinate geografiche
                    coord_key = (round(float(lat), 5), round(float(lon), 5))

                    # se questo campo non è già presente, lo aggiunge in coda
                    if coord_key not in existing_coordinates:
                        existing_coordinates.add(coord_key)
                        points.append({
                            "year": int(year),        # anno coltura
                            "lon": float(lon),        # longitudine
                            "lat": float(lat),        # latitudine
                            "code": int(crop_code)    # codice coltura
                        })
                        added_for_this_class += 1

print(f"\nEstrazione completata! Totale punti aggiornato: {len(points)}")

Campionamento in corso per l'anno 2023 (4 file)

Estrazione completata! Totale punti aggiornato: 1707


#### Salvataggio dei punti

In [5]:
import json
from pathlib import Path

# percorso del file JSON dei punti estratti
points_path = Path(f'{DATA_DIR}/interim/points.json')
points_path.parent.mkdir(parents=True, exist_ok=True)

with open(points_path, "w", encoding="utf-8") as f:
    json.dump(points, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(points)} punti in: {points_path.resolve()}")

✅ Salvati 1707 punti in: /content/drive/.shortcut-targets-by-id/1MZsG0ApsPUjw5Xpf76TSg5ONNZpYIZnf/Progetto MLDM/data/interim/points.json


#### Creazione e salvataggio legenda delle colture

In [6]:
import pandas as pd
import xml.etree.ElementTree as ET

# cerca i files .aux.xml che contengono i metadati
files = list(main_directory.rglob("*.aux.xml"))

if not files:
    raise FileNotFoundError(f"Nessun file .aux.xml trovato all'interno di: {main_directory}")

# prende il primo file trovato
file_path = files[0]
print(f"Trovato file .aux.xml: {file_path.name}")

tree = ET.parse(file_path)
root = tree.getroot()

legend = {}

# estrae il codice e il nome della coltura
for r in root.findall(".//Row"):
    fields = r.findall("F")
    
    crop_code = int(fields[0].text)     
    crop_name = fields[2].text
    
    legend[crop_code] = crop_name

Trovato file .aux.xml: CLMS_HRLVLCC_CTY_S2020_R10m_E47N20_03035_V01_R00.tif.aux.xml


In [7]:
import json
from pathlib import Path

# percorso del file JSON della legenda
legend_path = Path(f'{DATA_DIR}/processed/legend.json')

with open(legend_path, "w", encoding="utf-8") as f:
    json.dump(legend, f, indent=4, ensure_ascii=False)

print(f"✅ Salvate {len(legend)} voci nella legenda: {legend_path.resolve()}")

✅ Salvate 21 voci nella legenda: /content/drive/.shortcut-targets-by-id/1MZsG0ApsPUjw5Xpf76TSg5ONNZpYIZnf/Progetto MLDM/data/processed/legend.json


#### Riepilogo del campione di colture

In [8]:
import pandas as pd

df_points = pd.read_json(f'{DATA_DIR}/interim/points.json')

df_points['crop'] = df_points['code'].map(legend).fillna("unknown")

summary = df_points.groupby(['code', 'crop']).size().reset_index(name='n_points')                                                                               
summary['percentage (%)'] = ((summary['n_points'] / len(df_points)) * 100).round(1)                                                                             
summary = summary.sort_values(by='n_points', ascending=False).reset_index(drop=True)

print(f"Riepilogo campionamento (Totale punti: {len(df_points)}):")
display(summary)

Riepilogo campionamento (Totale punti: 1707):


,code,crop,n_points,percentage (%)
0,1110,Wheat,120,7.0
1,1120,Barley,120,7.0
2,1210,Fresh Vegetables,120,7.0
3,1310,Potatoes,120,7.0
4,1220,Dry Pulses,120,7.0
5,1430,Rapeseed,120,7.0
6,1410,Sunflower,120,7.0
7,2310,Fruits,120,7.0
8,2200,Olives,120,7.0
9,1440,Flax cotton and hemp,120,7.0
